In [2]:
import os
import json
import glob
import numpy as np
import os
# Define the folder path
folder_path = '/Users/vaibhavmishra/Desktop/clash_squad_agent_partitioned_features'

# Get list of all json files in the folder
json_files = glob.glob(os.path.join(folder_path, '*.json'))

# Load all json files
chunk_sizes = [500, 200, 100, 50]
save_dir = '/Users/vaibhavmishra/Desktop/clash_squad_agent_trajectories'
os.makedirs(save_dir, exist_ok=True)

def predict_movement(chunk):
    # Extract action features from each timestep in the chunk
    # Each timestep has format: [state_features, action_features]
    # action_features is a list of 10 values, first 2 might be movement
    actions = [timestep[1] for timestep in chunk]
    move_x = [abs(action[0]) for action in actions]
    move_y = [abs(action[1]) for action in actions]
    avg_move_x = sum(move_x) / len(move_x)
    avg_move_y = sum(move_y) / len(move_y)
    if avg_move_x > 0.5 and avg_move_y > 0.5:
        return 1
    return 0

        
def save_trajectory(chunk, count):
    # Convert chunk to a structured numpy array
    # Flatten each timestep into a single feature vector
    feature_list = [feature[0] for feature in chunk]
    # Generate unique filename
    filename = f'trajectory_{count}.npy'
    save_path = os.path.join(save_dir, filename)
    
    # Save as npy file
    np.save(save_path, feature_list)
    
def find_trajectories(data, count):
    
    i = 0
    while i < len(data['features']):
        # Try largest chunk size first
        valid_chunk_found = False
        for chunk_size in chunk_sizes:  # chunk_sizes is sorted largest to smallest
            if i + chunk_size > len(data['features']):
                continue
                
            chunk = data['features'][i:i+chunk_size]
            valid_movement = predict_movement(chunk)
            
            if valid_movement:
                count += 1
                save_trajectory(chunk, count)
                i += chunk_size
                valid_chunk_found = True
                break
        
        # If no valid chunk found at current position, move forward by 1
        if not valid_chunk_found:
            i += 1
                
    return count
        


# Process files
total_trajectories = 0
count = 0
from tqdm import tqdm

for file_path in tqdm(json_files, desc="Processing JSON files"):
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        count = find_trajectories(data, count)
        
print(f"Total trajectories saved: {count}")



Processing JSON files: 100%|██████████| 20497/20497 [1:22:08<00:00,  4.16it/s]

Total trajectories saved: 205588
